A notebook to process LLM outputs into familiar formats.

## Data Loading

In [3]:

from dataset_processing import process_llm_jsonl_results, CLIRENER_LABELS_V1, transform_to_ner_format
from EXPERIMENTS.evaluate_gold import load_and_merge_gold_data
import wandb
from EXPERIMENTS.evaluate import run_nervaluate, log_to_wandb

Based on [`run_campaign()`](EXPERIMENTS/evaluate_gold.py):

In [4]:
# True data loading
GOLD_DATASET_ID = "P0L3/CliReNER_v_1_1_28_GOLD_authorannots"
gold_data, bio_label_list = load_and_merge_gold_data(GOLD_DATASET_ID)
target_labels = list(CLIRENER_LABELS_V1)

# Predictions loading
llm_predictions_dir = "RESULTS/LLM_PREDICTIONS/ner_results_gpt_5_1.jsonl"
llm_name = llm_predictions_dir.split("/")[-1].replace("ner_results_", "").replace(".jsonl", "")
raw_predictions = process_llm_jsonl_results(llm_predictions_dir)
print(llm_name)

# WANDB initialization
WANDB_PROJECT = "CLIRENER_GOLD_SEEDS_authorannots"

# B. Initialize WandB Run
run_name = f"eval_GOLD_{llm_name}_zs"

# Start a fresh run for this evaluation
run = wandb.init(
    project=WANDB_PROJECT,
    name=run_name,
    reinit=True, # Allow multiple runs in one script
    config={
        "model_type": "LLM_"+llm_name,
        "model_id": llm_name,
        "training_dataset": "None",
        "evaluation_dataset": GOLD_DATASET_ID, # Explicitly logging this
        "seed": -1,
        "evaluation_scope": "ALL_SPLITS_MERGED"
    }
)

# D. Transform Predictions
print("--- Transforming Predictions ---")
model_predictions_transformed = transform_to_ner_format(raw_predictions, target_labels)

pred_lookup = {row['text']: row['ner_tags'] for row in model_predictions_transformed[0]}
true_ids = []
pred_ids = []
missing_count = 0

for row in gold_data["test"]:
    text_key = row['text']
    
    # 3. Match by exact text
    if text_key in pred_lookup:
        true_ids.append(row['ner_tags'])
        pred_ids.append(pred_lookup[text_key])
    else:
        missing_count += 1

if missing_count > 0:
    print(f"Warning: {missing_count} rows from Gold Data were not found in Predictions.")
print(f"aligned {len(true_ids)} rows for evaluation.")

# pred_ids = [row["ner_tags"] for row in model_predictions_transformed[0]]

# # Get True IDs from the merged dataset
# true_ids = gold_data["test"]["ner_tags"]

# E. Calculate Metrics
# We pass the dataset's BIO scheme for ID mapping, and the target labels for reporting
results, results_by_tag = run_nervaluate(true_ids, pred_ids, bio_label_list, tags=target_labels)

# F. Log to WandB
log_to_wandb(results, results_by_tag)

print(f"SUCCESS: {run_name}")

wandb.finish()

--- Loading and Merging GOLD Dataset: P0L3/CliReNER_v_1_1_28_GOLD_authorannots ---
Merged ['train', 'validation', 'test'] into a single dataset of size: 192
gpt_5_1


--- Transforming Predictions ---
aligned 192 rows for evaluation.
--- Calculating Metrics ---

--- Results Logged to WandB ---
              ent_type      partial       strict        exact
correct    1125.000000  1083.000000   807.000000  1083.000000
incorrect   614.000000     0.000000   932.000000   656.000000
partial       0.000000   656.000000     0.000000     0.000000
missed      628.000000   628.000000   628.000000   628.000000
spurious     36.000000    36.000000    36.000000    36.000000
possible   2367.000000  2367.000000  2367.000000  2367.000000
actual     1775.000000  1775.000000  1775.000000  1775.000000
precision     0.633803     0.794930     0.454648     0.610141
recall        0.475285     0.596113     0.340938     0.457541
f1            0.543216     0.681313     0.389667     0.522936
SUCCESS: eval_GOLD_gpt_5_1_zs


overall/exact_f1,▁
overall/exact_precision,▁
overall/exact_recall,▁
overall/partial_f1,▁
overall/partial_precision,▁
overall/partial_recall,▁
overall/strict_f1,▁
overall/strict_precision,▁
overall/strict_recall,▁
overall/type_f1,▁
+2,...
